# PFAS Groundwater — T2 Multilabel Baseline (Colab)

**Task T2**: predict which individual PFAS compounds exceed their per-compound threshold (multilabel).

This notebook is **AUTONOMOUS**: it bootstraps `src/`, downloads the dataset from Drive,
and runs 5 models: Prevalence floor, Binary Relevance, Masked Classifier Chain,
Ensemble Classifier Chains, and **FrequencyClassChain** (Dong et al. 2024-style cascade).

**Strict predictive mode**: no PFAS measurement is used as a feature (CLAUDE.md §1).

---
**PARAMETERS TO SET (cell below)**:
- `SMOKE_TEST` : `True` for a fast CPU sanity check (~3 min), `False` for the full GPU run.
- `BOOTSTRAP` : `"git"` to clone from GitHub, `"drive"` to copy `src/` from your Drive.
- `REPO_URL` / `GIT_REF` : GitHub repo URL and branch/commit (used when `BOOTSTRAP="git"`).
- `DRIVE_PROJECT_DIR` : absolute path on your mounted Drive.
- `DRIVE_DATA_PATH` : path on mounted Drive to `CA-PFAS-ASGWS.parquet`, **OR** set `GDRIVE_FILE_ID`.

In [ ]:
# ============================================================
#  USER PARAMETERS — edit these before running
# ============================================================

SMOKE_TEST = True          # True = fast CPU sanity check; False = full GPU run

# --- Code bootstrap mode ---
# "git"   -> git clone REPO_URL@GIT_REF (needs up-to-date code on remote)
# "drive" -> copy src/ from DRIVE_PROJECT_DIR/src/
BOOTSTRAP = "git"
REPO_URL  = "https://github.com/dnwiloic/pfas-gnn.git"
GIT_REF   = "main"   # branch name or full commit SHA

# --- Drive paths ---
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/pfas-gnn"   # <-- SET THIS

# --- Dataset (parquet file) ---
# Option A: path on mounted Drive
DRIVE_DATA_PATH = "/content/drive/MyDrive/pfas-gnn/data/CA-PFAS-ASGWS.parquet"  # <-- SET THIS
# Option B: gdown (set GDRIVE_FILE_ID to non-empty string to use gdown instead)
GDRIVE_FILE_ID = ""   # e.g. "1AbCdEfGh..." from the Google Drive share URL

print(f"SMOKE_TEST={SMOKE_TEST}  BOOTSTRAP={BOOTSTRAP!r}")

## Cell 1 — GPU detection & version info

In [ ]:
import sys, platform
print(f"Python : {sys.version}")
print(f"Platform: {platform.platform()}")

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    import subprocess
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if r.returncode == 0:
        print("\n[GPU detected]")
        print(r.stdout[:800])
    else:
        print("[WARNING] No GPU found.")
        print("Note: T2 baselines are CPU-bound (sklearn HGB). A high-RAM CPU instance")
        print("is likely more efficient than a GPU instance for this run.")
        print(r.stderr[:200])
else:
    print("[Local run — no GPU check performed]")

try:
    import numpy as np; print(f"numpy   : {np.__version__}")
except ImportError: print("numpy   : NOT INSTALLED")
try:
    import sklearn; print(f"sklearn : {sklearn.__version__}")
except ImportError: print("sklearn : NOT INSTALLED")
try:
    import pandas as pd; print(f"pandas  : {pd.__version__}")
except ImportError: print("pandas  : NOT INSTALLED")

## Cell 2 — Install dependencies (Colab only)

Note: PyTorch Geometric is NOT needed for these non-graph baselines.

In [ ]:
if IN_COLAB:
    import subprocess, sys
    pkgs = [
        "scikit-learn>=1.4,<2.0",
        "imbalanced-learn>=0.11,<1.0",
        "pyyaml>=6.0,<7.0",
        "pandas>=2.0,<3.0",
        "pyarrow>=14.0",
        "scipy>=1.11",
        "numpy>=1.24",
    ]
    print("Installing packages...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr[-1000:])
        raise RuntimeError("Dependency installation failed.")
    print("Packages installed. Verifying imports...")
    import imblearn, yaml, pandas, sklearn
    print(f"  sklearn={sklearn.__version__}  imblearn={imblearn.__version__}")
    print("Imports OK.")
else:
    print("[Local run] Skipping pip install — using local environment.")

## Cell 3 — Bootstrap `src/` code

**CRITICAL**: the remote must contain up-to-date `src/baselines_t2.py` (with `FrequencyClassChain`)
and `src/metrics.py`. If these are not yet pushed, push them first or use `BOOTSTRAP="drive"`.

In [ ]:
import sys, shutil
from pathlib import Path

REPO_LOCAL = Path("/content/pfas-gnn") if IN_COLAB else Path("..").resolve()

if IN_COLAB:
    if BOOTSTRAP == "git":
        import subprocess
        if REPO_LOCAL.exists():
            print(f"Repo already cloned at {REPO_LOCAL} — pulling latest...")
            subprocess.run(["git", "-C", str(REPO_LOCAL), "fetch", "origin"],
                           capture_output=True, text=True)
            subprocess.run(["git", "-C", str(REPO_LOCAL), "checkout", GIT_REF],
                           capture_output=True, text=True)
            r3 = subprocess.run(["git", "-C", str(REPO_LOCAL), "reset", "--hard",
                                 f"origin/{GIT_REF}"],
                                capture_output=True, text=True)
            print(r3.stdout or r3.stderr)
        else:
            print(f"Cloning {REPO_URL} @ {GIT_REF} ...")
            r = subprocess.run(
                ["git", "clone", "--depth=1", "--branch", GIT_REF, REPO_URL, str(REPO_LOCAL)],
                capture_output=True, text=True
            )
            if r.returncode != 0:
                print("[ERROR] git clone failed:")
                print(r.stderr)
                raise RuntimeError("git clone failed. Check REPO_URL / GIT_REF.")
            print("Cloned OK.")
    elif BOOTSTRAP == "drive":
        src_drive = Path(DRIVE_PROJECT_DIR) / "src"
        dst_src   = REPO_LOCAL / "src"
        if not src_drive.exists():
            raise FileNotFoundError(
                f"src/ not found on Drive at {src_drive}.\n"
                "Copy pfas-gnn/src/ to your Drive first."
            )
        if dst_src.exists():
            shutil.rmtree(dst_src)
        shutil.copytree(str(src_drive), str(dst_src))
        print(f"Copied src/ from Drive -> {dst_src}")
    else:
        raise ValueError(f"Unknown BOOTSTRAP mode: {BOOTSTRAP!r}. Use 'git' or 'drive'.")
else:
    REPO_LOCAL = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path("..").resolve()
    print(f"[Local run] Using repo at {REPO_LOCAL}")

repo_str = str(REPO_LOCAL)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
print(f"sys.path[0] = {sys.path[0]}")

In [ ]:
# --- Guard-rail: verify code is up-to-date ---
import sys

for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

try:
    import src.baselines_t2
    import src.baselines_t1
    import src.metrics
except ImportError as e:
    raise ImportError(
        f"CRITICAL: Cannot import src modules: {e}\n"
        "If BOOTSTRAP='git': push baselines_t1.py, baselines_t2.py, metrics.py to the remote.\n"
        "If BOOTSTRAP='drive': ensure src/ on Drive is up-to-date."
    )

assert hasattr(src.baselines_t2, "FrequencyClassChain"), (
    "CRITICAL: FrequencyClassChain not found in src.baselines_t2.\n"
    "The code on Colab is OBSOLETE.\n"
    "Action: push the latest baselines_t2.py to the remote (git) OR "
    "copy src/ with the up-to-date files to your Drive (drive)."
)

from src.metrics import multilabel_metrics, REQUIRED
assert set(REQUIRED) == {"roc_auc", "f1", "accuracy", "recall", "precision"}, (
    "CRITICAL: src.metrics.REQUIRED is wrong. Code is obsolete."
)

print("[guard-rail] src/ is up-to-date: FrequencyClassChain found, metrics OK.")
import src.baselines_t2 as B
import src.config as C
import src.data as D
import src.splits as S
import src.features as F
print("[guard-rail] All src modules imported.")

## Cell 4 — Mount Drive & load dataset

In [ ]:
import os
from pathlib import Path

DATA_PATH = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("Drive mounted.")

    if GDRIVE_FILE_ID:
        import subprocess
        dest = Path("/content/CA-PFAS-ASGWS.parquet")
        if not dest.exists():
            print(f"Downloading dataset via gdown (ID={GDRIVE_FILE_ID})...")
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "gdown"],
                capture_output=True, text=True
            )
            import gdown
            gdown.download(id=GDRIVE_FILE_ID, output=str(dest), quiet=False)
        else:
            print(f"Dataset already at {dest}.")
        DATA_PATH = dest
    else:
        DATA_PATH = Path(DRIVE_DATA_PATH)
        if not DATA_PATH.exists():
            raise FileNotFoundError(
                f"Dataset not found at {DATA_PATH}.\n"
                "Set DRIVE_DATA_PATH to the parquet on your mounted Drive, "
                "or set GDRIVE_FILE_ID to download via gdown."
            )
        print(f"Dataset path: {DATA_PATH}  size={DATA_PATH.stat().st_size/1e6:.1f} MB")
else:
    DATA_PATH = REPO_LOCAL / "data" / "CA-PFAS-ASGWS.parquet"
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Local dataset not found at {DATA_PATH}. "
            "Ensure data/CA-PFAS-ASGWS.parquet exists."
        )
    print(f"[Local] Dataset: {DATA_PATH}  size={DATA_PATH.stat().st_size/1e6:.1f} MB")

# Integrity check
import pandas as pd
print("Loading parquet for integrity check...")
_df_check = pd.read_parquet(DATA_PATH)
n_rows, n_cols = _df_check.shape
print(f"Shape: {n_rows} x {n_cols}")

REQUIRED_COLS = ["gm_well_id", "latitude", "longitude", "PFOA_ngL"]
missing_cols = [c for c in REQUIRED_COLS if c not in _df_check.columns]
if missing_cols:
    raise ValueError(
        f"INTEGRITY FAIL: columns {missing_cols} missing from the parquet.\n"
        "Check that DRIVE_DATA_PATH / GDRIVE_FILE_ID points to the correct file."
    )
if n_cols != 201:
    print(f"[WARNING] Expected 201 columns, got {n_cols}. Check dataset version.")
if n_rows != 46338:
    print(f"[WARNING] Expected 46338 rows, got {n_rows}. This may be an older snapshot.")
if n_cols == 201 and n_rows == 46338:
    print("[INTEGRITY OK] shape==(46338, 201), required columns present.")
del _df_check

# Override DATA_PARQUET in config
C.DATA_PARQUET = DATA_PATH
print(f"src.config.DATA_PARQUET -> {C.DATA_PARQUET}")

## Cell 5 — Configure run parameters

In [ ]:
from pathlib import Path
import numpy as np

# Artifact output directory
if IN_COLAB:
    SAVE_DIR = Path(DRIVE_PROJECT_DIR) / "experiments" / f"baseline_t2{'_smoke' if SMOKE_TEST else ''}"
else:
    SAVE_DIR = REPO_LOCAL / "experiments" / f"baseline_t2{'_smoke' if SMOKE_TEST else ''}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be saved to: {SAVE_DIR}")

# Run configuration
LABELS      = list(C.T2_LABELS)                   # 10 labels (9 core + PFNA)
FEATURE_COLS = C.feature_columns(include_location=False, cocontam="core", include_air=False)
K           = 2 if SMOKE_TEST else 8              # spatial CV folds
SMOKE_N     = 800                                  # wells for smoke mode
SMALL       = SMOKE_TEST                           # small HGB models in smoke
MAX_ITER    = 60 if SMOKE_TEST else 200

print(f"Labels: {LABELS}")
print(f"Features: {len(FEATURE_COLS)} columns")
print(f"K={K}  SMALL={SMALL}  MAX_ITER={MAX_ITER}  SMOKE_N={SMOKE_N if SMOKE_TEST else 'all'}")

## Cell 6 — Load data & build splits

In [ ]:
import time

t_load = time.time()
df = D.load(smoke=SMOKE_TEST, smoke_n=SMOKE_N)
print(f"[data] rows={len(df)}  wells={df[C.WELL_ID].nunique()}  "
      f"labels={len(LABELS)}  feats={len(FEATURE_COLS)}  SMOKE={SMOKE_TEST}")

# Verify no feature leakage
leak = set(FEATURE_COLS) & set(C.LEAKAGE_BLOCKLIST)
assert not leak, f"LEAKAGE detected in features: {leak}"
print("[leak check] OK — no PFAS-derived column in feature set.")

# Build folds
spatial = S.spatial_block_folds(df, k=K)
random  = S.group_random_folds(df, k=K)
S.assert_no_group_leak(df, spatial)
S.assert_no_group_leak(df, random)
print(f"[splits] spatial k={len(np.unique(spatial))}  random k={len(np.unique(random))}")

# Measurement mask summary
Y, M = B.masked_targets(df, labels=LABELS)
print("[mask] measured %:")
for a in LABELS:
    pct = M[f"label_{a}"].mean() * 100
    prev = float(Y[f"label_{a}"].to_numpy()[M[f"label_{a}"].to_numpy()].mean()) * 100
    print(f"  {a:10s}: measured={pct:.1f}%  prevalence={prev:.1f}%")

# Patch max_iter without touching src defaults
import src.baselines_t2 as _bt
_orig_make = _bt.make_estimator
def _patched_make(kind="hgb", *, class_weight=None, small=False):
    est = _orig_make(kind, class_weight=class_weight, small=small)
    if hasattr(est, "max_iter") and not small:
        est.set_params(max_iter=MAX_ITER)
    return est
_bt.make_estimator = _patched_make
print(f"[patch] HGB max_iter set to {MAX_ITER} for non-small models.")

## Cell 7 — Run 5 models (with per-model checkpointing)

**Smoke** (`SMOKE_TEST=True`): ~800 wells, 2 folds, small models — **CPU < 3 min**.

**Full run** (`SMOKE_TEST=False`): all 46 338 rows, 8 folds, 5 models, inner-CV chains.
Estimated duration: **~60–180 min CPU** (high variance due to inner-CV per label in chains).
Run on a Colab High-RAM instance or multi-core CPU box; expected **~30–90 min on Colab**.

Per-model checkpointing: `metrics_incremental.json` is updated after each model so that
a Colab disconnection does not lose completed models.

In [ ]:
import json, time

ORDER = tuple(LABELS)

def f_prev():   return B.PrevalenceBaseline(labels=LABELS)
def f_br():     return B.BinaryRelevance(kind="hgb", labels=LABELS,
                                         class_weight="balanced", small=SMALL)
def f_chain():  return B.MaskedClassifierChain(kind="hgb", order=ORDER,
                                               out_labels=ORDER, class_weight="balanced",
                                               small=SMALL, inner_k=2 if SMOKE_TEST else 3)
def f_ecc():    return B.EnsembleClassifierChains(kind="hgb",
                                                  n_chains=2 if SMOKE_TEST else 3,
                                                  labels=list(LABELS),
                                                  class_weight="balanced", small=SMALL)
def f_fcc():    return B.FrequencyClassChain(kind="hgb", labels=list(LABELS),
                                             n_classes=4, class_weight="balanced",
                                             small=SMALL,
                                             inner_k=2 if SMOKE_TEST else 3)

MODEL_SPECS = [
    ("Prevalence",       f_prev,  False),
    ("BinaryRelevance",  f_br,    False),
    ("Chain",            f_chain, True),
    ("Ensemble",         f_ecc,   True),
    ("FreqClassChain",   f_fcc,   True),
]

results = {}
t_all = time.time()
ckpt_path = SAVE_DIR / "metrics_incremental.json"

for nm, fac, use_groups in MODEL_SPECS:
    t1 = time.time()
    print(f"\n[{nm}] running spatial CV (k={K})...")
    sp = B.evaluate_model(df, fac, spatial, FEATURE_COLS, labels=LABELS,
                          use_groups=use_groups)
    print(f"[{nm}] running random CV (k={K})...")
    rd = B.evaluate_model(df, fac, random, FEATURE_COLS, labels=LABELS,
                          use_groups=use_groups)
    results[nm] = {"spatial": sp, "random": rd}
    a, b = sp["aggregate"], rd["aggregate"]
    elapsed_m = time.time() - t1
    print(f"  [{nm}] DONE in {elapsed_m:.0f}s")
    print(f"    spatial: macroAUROC={a['macro_AUROC']:.3f}  microF1={a['micro_F1']:.3f}  "
          f"Hamming={a['Hamming']:.3f}  EMR={a['EMR']:.3f}")
    print(f"    random : macroAUROC={b['macro_AUROC']:.3f}  "
          f"ΔAUROC={b['macro_AUROC']-a['macro_AUROC']:+.3f}")

    # --- Checkpoint: save incremental results ---
    def _pack_lite(res):
        """Lightweight serialisable summary (no oof_P arrays)."""
        return {"aggregate": res["aggregate"],
                "thresholds": res["thresholds"],
                "per_label": res["per_label"].to_dict(orient="records"),
                "per_fold": res["per_fold"].to_dict(orient="records"),
                "spread": res["spread"]}
    ckpt = {nm_: {"spatial": _pack_lite(results[nm_]["spatial"]),
                  "random": _pack_lite(results[nm_]["random"])}
            for nm_ in results}
    ckpt_path.write_text(json.dumps(ckpt, indent=2, default=float))
    print(f"  [checkpoint] saved to {ckpt_path}")

print(f"\nAll models done in {time.time()-t_all:.0f}s")

## Cell 8 — SMOTE ablation on PFNA (rare label)

In [ ]:
import time

t_smote = time.time()
print("Running SMOTE ablation on PFNA (rare regulated label)...")

def f_br_smote():
    return B.BinaryRelevance(kind="hgb", labels=LABELS, class_weight="balanced",
                             smote_labels=("PFNA",), small=SMALL)

smote_res = B.evaluate_model(df, f_br_smote, spatial, FEATURE_COLS, labels=LABELS)
pfna_pl_cw = results["BinaryRelevance"]["spatial"]["per_label"]
pfna_cw = float(pfna_pl_cw.loc[pfna_pl_cw.label == "PFNA", "AUROC"].iloc[0])
pfna_sm_pl = smote_res["per_label"]
pfna_sm = float(pfna_sm_pl.loc[pfna_sm_pl.label == "PFNA", "AUROC"].iloc[0])

print(f"PFNA AUROC: class_weight={pfna_cw:.3f}  +SMOTE={pfna_sm:.3f}  "
      f"delta={pfna_sm-pfna_cw:+.3f}  ({time.time()-t_smote:.0f}s)")

## Cell 9 — Pseudo-label probe (semi-supervision on reduced-panel labels)

In [ ]:
import time

t_pseudo = time.time()
print("Running pseudo-label probe (PFBA, PFPeA, PFPeS)...")
probe = B.pseudo_label_probe(df, spatial, FEATURE_COLS,
                             target_labels=("PFBA", "PFPeA", "PFPeS"), small=SMALL)
print(f"Probe done in {time.time()-t_pseudo:.0f}s")
if len(probe):
    print(probe.to_string(index=False, float_format="{:.3f}".format))
else:
    print("(no fold met the size guards in smoke mode -> expected, runs on full data)")

## Cell 10 — Paired comparison BR vs Chain (Wilcoxon)

In [ ]:
paired = {}
for metric in ["macro_AUROC", "micro_F1"]:
    md, p = B.paired_compare(
        results["BinaryRelevance"]["spatial"]["per_fold"],
        results["Chain"]["spatial"]["per_fold"],
        metric=metric
    )
    paired[metric] = {"chain_minus_br": md, "wilcoxon_p": p}
    sig = "*" if (p is not None and not np.isnan(p) and p < 0.05) else ""
    print(f"[paired] {metric}: chain-BR = {md:+.4f}  Wilcoxon p = {p}{sig}")

## Cell 11 — Results table: models × 5 metrics (spatial + random + Δ)

In [ ]:
import pandas as pd
import numpy as np

print("\n" + "="*90)
print("MODELS x 5 METRICS (micro-averaged) — T2 multilabel baseline")
print("="*90)

rows = []
for nm in ["Prevalence", "BinaryRelevance", "Chain", "Ensemble", "FreqClassChain"]:
    if nm not in results:
        continue
    sp = results[nm]["spatial"]["aggregate"]
    rd = results[nm]["random"]["aggregate"]
    rows.append({
        "model":           nm,
        # 5 required metrics — spatial CV (reference)
        "macro AUROC (sp)": sp["macro_AUROC"],
        "micro F1 (sp)":   sp["micro_F1"],
        "micro Acc (sp)":  sp["micro_accuracy"],
        "micro Rec (sp)":  sp["micro_recall"],
        "micro Prec (sp)": sp["micro_precision"],
        # multilabel extras
        "Hamming (sp)":    sp["Hamming"],
        "EMR (sp)":        sp["EMR"],
        # random + delta
        "macro AUROC (rd)": rd["macro_AUROC"],
        "Δ AUROC":         rd["macro_AUROC"] - sp["macro_AUROC"],
    })

df_res = pd.DataFrame(rows).set_index("model")
pd.options.display.float_format = "{:.3f}".format
print(df_res.to_string())

print("\nNote: macro AUROC = unweighted mean over labels (threshold-free).")
print("      micro metrics = pooled over all (row, label) measured cells.")
print("      Δ AUROC = random - spatial (positive -> spatial autocorrelation artefact).")

## Cell 12 — FrequencyClassChain: 4 frequency classes

In [ ]:
# Show the 4 frequency classes used by FrequencyClassChain (Dong et al. 2024 style)
print("FrequencyClassChain — 4 frequency classes (from least rare to rarest)")
print("(Classes determined by prevalence on the full dataset measured rows)\n")

import numpy as np
Yf, Mf = B.masked_targets(df, labels=LABELS)
freq = {a: (float(Yf[f"label_{a}"].to_numpy()[Mf[f"label_{a}"].to_numpy()].mean())
            if Mf[f"label_{a}"].any() else 0.0) for a in LABELS}
order = sorted(LABELS, key=lambda a: -freq[a])
classes = [list(g) for g in np.array_split(np.array(order, dtype=object), 4) if len(g)]

for i, cl in enumerate(classes, 1):
    print(f"  Class {i} ({['least rare','...','...','rarest'][i-1]}): "
          + ", ".join(f"{a} (prev={freq[a]:.3f})" for a in cl))

print()
print("Chain runs in frequency order: each label is predicted from X + all more-frequent")
print("labels already predicted (Dong et al. 2024 cascade approach, leak-free at train).")

## Cell 13 — Per-label results (Binary Relevance vs Chain, spatial CV)

In [ ]:
import pandas as pd

print("PER-LABEL RESULTS — Binary Relevance (spatial CV, OOF thresholds)")
print("5 metrics per label: AUROC / F1 / accuracy / recall / precision\n")

pl_br = results["BinaryRelevance"]["spatial"]["per_label"].copy()
pl_ch = results["Chain"]["spatial"]["per_label"].set_index("label").copy()

pl_br["Chain AUROC"] = [float(pl_ch.loc[a, "AUROC"]) if a in pl_ch.index else float("nan")
                        for a in pl_br["label"]]
pl_br["delta AUROC (Chain-BR)"] = pl_br["Chain AUROC"] - pl_br["AUROC"]

display_cols = ["label", "n_measured", "prevalence",
                "AUROC", "Chain AUROC", "delta AUROC (Chain-BR)",
                "f1", "accuracy", "recall", "precision"]
pl_show = pl_br[[c for c in display_cols if c in pl_br.columns]]
pd.options.display.float_format = "{:.3f}".format
print(pl_show.to_string(index=False))

## Cell 14 — Save all artifacts to Drive

In [ ]:
import json, yaml, time

t_save = time.time()

# Config
cfg = {
    "task": "T2_multilabel_baseline",
    "seed": int(C.SEED),
    "smoke": SMOKE_TEST,
    "labels": list(LABELS),
    "n_feature_cols": len(FEATURE_COLS),
    "feature_cols": list(FEATURE_COLS),
    "cv": {"spatial_k": int(K), "random_k": int(K), "group_key": C.WELL_ID},
    "models": [nm for nm, _, _ in MODEL_SPECS],
    "hgb_max_iter": MAX_ITER,
    "encode": "frequency",
    "threshold": "per-label F1 on OOF probabilities",
    "imbalance": "class_weight=balanced; SMOTE ablation on PFNA",
}
(SAVE_DIR / "config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

# Full metrics
def pack(res):
    return {
        "aggregate":  res["aggregate"],
        "spread":     res["spread"],
        "thresholds": res["thresholds"],
        "per_label":  res["per_label"].to_dict(orient="records"),
        "per_fold":   res["per_fold"].to_dict(orient="records"),
    }

metrics = {
    "smoke": SMOKE_TEST,
    "seed":  int(C.SEED),
    "models": {nm: {"spatial": pack(results[nm]["spatial"]),
                    "random":  pack(results[nm]["random"])}
               for nm in results},
    "smote_ablation_PFNA": {"class_weight": pfna_cw, "smote": pfna_sm},
    "paired_br_vs_chain": paired,
    "pseudo_label_probe": probe.to_dict(orient="records") if len(probe) else [],
}
(SAVE_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))

# Per-label table as CSV
results["BinaryRelevance"]["spatial"]["per_label"].to_csv(
    SAVE_DIR / "per_label_BR_spatial.csv", index=False
)

print(f"Artifacts saved in {time.time()-t_save:.1f}s to: {SAVE_DIR}")
for f in sorted(SAVE_DIR.iterdir()):
    print(f"  {f.name:45s}  {f.stat().st_size/1024:.1f} KB")

## Cell 15 — Timing summary & full-run estimate

In [ ]:
import time

elapsed_total = time.time() - t_all
print(f"Total run time: {elapsed_total:.1f}s ({elapsed_total/60:.2f} min)")

if SMOKE_TEST:
    # Extrapolation: scale by rows, labels, folds
    n_full = 46338
    n_smoke = len(df)
    scale_rows  = n_full / n_smoke
    scale_folds = 8 / K
    scale_labels = len(C.T2_LABELS) / 7     # smoke uses fewer labels in some test modes
    scale_model = 3.5   # non-small HGB is ~3-4x slower
    lo = elapsed_total * scale_rows * scale_folds * scale_model / 60
    hi = lo * 1.5
    print(f"\nEXTRAPOLATED full run: ~{lo:.0f}-{hi:.0f} min CPU")
    print(f"  (x{scale_rows:.1f} rows, x{scale_folds:.0f} folds, ~x{scale_model} model size)")
    print(f"  On Colab (High-RAM CPU instance or GPU): expect ~30-90 min.")
    print()
    print("SMOKE TEST PASSED. To run the full pipeline, set SMOKE_TEST=False.")
    print("Checkpoints saved per model -> Colab disconnection will not lose completed models.")
else:
    print("FULL RUN COMPLETED.")
    print(f"Results in: {SAVE_DIR}")